<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/35_strategy_library_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Strategy Library (Baselines) ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import math, shutil

# Paths
PROJECT_DIR = Path("/content/drive/MyDrive/gt-markets")
DATA_DIR    = PROJECT_DIR/"data"/"processed"
ARTE_ROOT   = PROJECT_DIR/"AppDemo"/"artefacts"

# Assets & settings
ASSETS = {
    "GOLD":   "GC=F Close",
    "BTC":    "BTC-USD Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}
FREQS = ["D","W"]
ANNUALIZATION = {"D":252,"W":52}
COST_BPS = 5.0

# Load latest processed dataset
rawfile = sorted(DATA_DIR.glob("merged_financial_trends_data_*.csv"))[-1]
df_all = pd.read_csv(rawfile, parse_dates=["Date"]).set_index("Date").sort_index()

def to_frequency(df, freq):
    if freq=="D": return df
    return df.resample("W").last()

def add_indicators(px: pd.Series):
    d = pd.DataFrame(index=px.index)
    d["Close"] = px
    d["SMA20"] = px.rolling(20).mean()
    d["SMA50"] = px.rolling(50).mean()
    delta = px.diff()
    up = np.where(delta>0, delta, 0); dn = np.where(delta<0, -delta, 0)
    roll_up = pd.Series(up,index=px.index).rolling(14).mean()
    roll_dn = pd.Series(dn,index=px.index).rolling(14).mean()
    rs = roll_up/roll_dn.replace(0,np.nan); d["RSI14"] = 100 - (100/(1+rs))
    ma = px.rolling(20).mean(); sd = px.rolling(20).std()
    d["BB_mid"]=ma; d["BB_low"]=ma-2*sd; d["BB_high"]=ma+2*sd
    d["Donch_hi20"] = px.rolling(20).max()
    d["Donch_lo10"] = px.rolling(10).min()
    return d.dropna()

# Strategies
def S1_trend(df): return (df["SMA20"]>df["SMA50"]).astype(float)
def S2_meanrev(df):
    px=df["Close"]
    enter=(px<df["BB_low"])&(df["RSI14"]<30)
    exit_=(px>=df["BB_mid"])|(df["RSI14"]>50)
    pos=pd.Series(0.0,index=df.index); in_pos=False
    for i in range(len(df)):
        if not in_pos and enter.iloc[i]: in_pos=True
        elif in_pos and exit_.iloc[i]: in_pos=False
        pos.iloc[i]=1 if in_pos else 0
    return pos
def S3_breakout(df):
    px=df["Close"]
    enter=px>df["Donch_hi20"].shift(1)
    exit_=px<df["Donch_lo10"].shift(1)
    pos=pd.Series(0.0,index=df.index); in_pos=False
    for i in range(len(df)):
        if not in_pos and enter.iloc[i]: in_pos=True
        elif in_pos and exit_.iloc[i]: in_pos=False
        pos.iloc[i]=1 if in_pos else 0
    return pos

STRATS={"S1_trend":S1_trend,"S2_meanrev":S2_meanrev,"S3_breakout":S3_breakout}

def backtest(px,pos,freq):
    ret=px.pct_change().fillna(0.0)
    pos=pos.fillna(0).clip(0,1)
    trades=pos.diff().abs().fillna(0.0)
    cost=trades*(COST_BPS/1e4)
    strat=(pos.shift(1)*ret)-cost
    eq=(1+strat).cumprod()
    ann=ANNUALIZATION[freq]
    mu=strat.mean()*ann
    sd=strat.std()*math.sqrt(ann) if strat.std()>0 else np.nan
    sharpe=mu/sd if sd and sd>0 else np.nan
    maxdd=(eq/eq.cummax()-1).min()
    return {"Return_Ann":mu,"Vol_Ann":sd,"Sharpe":sharpe,"MaxDD":maxdd}, eq

# Build artefacts
for freq in FREQS:
    dff = to_frequency(df_all, freq)
    for asset, col in ASSETS.items():
        if col not in dff.columns: continue
        df_i = add_indicators(dff[col].dropna())
        out_dir = ARTE_ROOT/asset; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
        rows=[]
        for strat_id,fn in STRATS.items():
            pos=fn(df_i)
            metrics,eq=backtest(df_i["Close"],pos,freq)
            # signals
            sig=pos.diff().fillna(0).replace({1.0:1,-1.0:0}).astype(int)
            pd.DataFrame({"Date":df_i.index,"signal":sig,"position":pos.astype(int),
                          "Close":df_i["Close"],"strategy":strat_id}).to_csv(
                out_dir/f"signals_{strat_id}_{freq}.csv", index=False)
            # fig
            figpath=out_dir/"figs"/f"{strat_id}_{freq}.png"
            ax=eq.plot(figsize=(6,3),title=f"{asset}-{strat_id}-{freq}"); ax.grid(True,alpha=0.3)
            ax.get_figure().savefig(figpath,dpi=150,bbox_inches="tight"); plt.close(ax.get_figure())
            # metrics
            rows.append({"asset":asset,"freq":freq,"strategy":strat_id,**metrics})
        pd.DataFrame(rows).to_csv(out_dir/f"metrics_baseline_{freq}.csv", index=False)

print("✅ Baseline strategies exported to", ARTE_ROOT)


Mounted at /content/drive
✅ Baseline strategies exported to /content/drive/MyDrive/gt-markets/AppDemo/artefacts
